# EOSC 213 – Homework 04
## Problem 5: Fitting a Nonlinear Ten-Parameter Model to Data 

In this notebook, we solve the inverse problem

$$
y'(t)
=
-0.45\,y(t)
+
\tanh\!\left(
\sum_{k=1}^{5}
\big[a_k\sin(0.7kt)+b_k\cos(0.7kt)\big]
\right),
\qquad
y(0)=y_0.
$$

The unknowns are the ten coefficients

$$
(a_1,b_1,a_2,b_2,a_3,b_3,a_4,b_4,a_5,b_5).
$$

The goal is to estimate these ten parameters from noisy observations stored in `data_problem05.pt`.

### What this notebook will do

We will:

1. load and visualize the data,
2. implement the nonlinear ODE model in PyTorch,
3. solve the ODE using Forward Euler with step size $h=0.02$,
4. define a least-squares loss,
5. optimize all ten parameters with a PyTorch optimizer,
6. plot the loss versus iteration,
7. compare the fitted model with the data,
8. print the learned coefficients,
9. plot the recovered forcing function,
10. explain why grid search is impractical here.


### Step 0 — Import libraries

We use:
- **torch** for writing the codes
- **matplotlib** for plotting

In [ ]:
# import necessary libraries
import torch
import matplotlib.pyplot as plt

### Step 1 — Load the observed data

The file `data_problem05.pt` contains:

- `t_obs`: observation times,
- `y_obs`: noisy observations,
- `y0`: initial condition.

Load these quantities to Python variables from the file. 

In [ ]:
# TODO 1: load the data from the file and extract t_obs, y_obs and the initial value y0 from it.
# Hint: use torch.load() to load the data from the .pt file

data = ... 
t_obs = ... 
y_obs = ... 
y0 = ...

print("Keys in file:", list(data.keys()))
print("Number of observations:", len(t_obs))
print("Initial condition y0 =", y0.item())

### Step 2 — Plot the observations

Before fitting anything, always plot the data first. This helps you see the time interval, the rough shape of the signal, and the level of noise.


In [ ]:
# This code block is for visualizing the observed data
plt.figure(figsize=(8, 5))
plt.plot(t_obs, y_obs, "o", label="Observed data")
plt.xlabel("t", fontsize=13)
plt.ylabel("y", fontsize=13)
plt.title("Observed noisy data", fontsize=15)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.show()


### Step 3 — Build the forcing function

The forcing term inside the ODE is

$$
g(t)=\sum_{k=1}^{5}\big[a_k\sin(0.7kt)+b_k\cos(0.7kt)\big].
$$

The unknown coefficients are the five `a_k` values and the five `b_k` values.

Write this forcing function as a Python function `forcing_function()` so that it can be reused inside the solver and later when plotting the recovered forcing function.


In [ ]:
# TODO 2: implement the forcing function g(t) as a function of time t and 
# parameters list a and b each containing 5 parameters. The function should 
# return a tensor of the same shape as t. 
def forcing_function(t, a, b):
    return NotImplementedError # remove this line and implement the function

### Step 4 — Implement the Forward Euler solver

We are told to use Forward Euler with step size

$$
h=0.02.
$$

For a general ODE

$$
y'(t)=f(t,y),
$$

Forward Euler updates via

$$
y_{n+1}=y_n+h\,f(t_n,y_n).
$$

In our problem,

$$
f(t,y)= -0.45\,y + \tanh(g(t)).
$$

We simulate on the full Euler grid, then extract the values at the observation times.


In [ ]:
h = 0.02

def simulate_model(t_obs, y0, a, b, h=0.02):
    t_start = t_obs[0].item()
    t_end = t_obs[-1].item()
    t_grid = torch.arange(t_start, t_end + 1e-9, h, dtype=t_obs.dtype)

    y_grid = torch.zeros_like(t_grid)
    y_grid[0] = y0

    for n in range(len(t_grid) - 1):
        t_n = t_grid[n]
        g_n = forcing_function(t_n.unsqueeze(0), a, b).squeeze(0)
        rhs = -0.45 * y_grid[n] + torch.tanh(g_n)
        y_grid[n + 1] = y_grid[n] + h * rhs

    indices = torch.round((t_obs - t_start) / h).long()
    y_pred = y_grid[indices]
    return y_pred, t_grid, y_grid


### Step 5 — Define the least-squares loss

To measure the mismatch between model and data, we use the mean squared error

$$
L = \frac{1}{N}\sum_{i=1}^N (y_i^{\mathrm{obs}}-y_i^{\mathrm{pred}})^2.
$$

A smaller loss means a better fit. Define the loss function `loss_fn()` below. 


In [ ]:
# TODO 3: define the loss function as the mean squared error between the observed data y_obs 
# and the model predictions y_pred. Within the loss_fn(), compute the model predictions 
# by calling the simulate_model function defined above with the current parameters a and b.

def loss_fn(a, b, t_obs, y_obs, y0):
    return NotImplementedError # remove this line and implement the function

### Step 6 — Initialize the unknown parameters

This is a ten-parameter problem, so a brute-force search is not realistic.

Instead, initialize the unknown coefficients near zero and let a PyTorch optimizer update them automatically.

Here, initialize the list of parameters `a` and `b` as tensors of zeros of dtype `float32` with the `requires_grad=True` flag so PyTorch can compute derivatives of the loss with respect to all these ten parameters.

After that, define the `Adam` optimizer with a suitable learning rate (you will have to play with the learning rate value several times to get a decreasing loss function). A learning rate that is too high might cause the loss to diverge, while a learning rate that is too small will make the parameter updates very slow, requiring really large number of parameter updates to reach a suitably low value of loss.

In [ ]:
# TODO 4: initialize the parameters a and b as tensors of zeros with requires_grad=True.
# Then, set up the Adam optimizer to optimize these parameters with a suitable learning rate.

torch.manual_seed(0)

a = ... 
b = ... 

optimizer = ... 

print("Initial a:", a.detach())
print("Initial b:", b.detach())

### Step 7 — Run gradient-based optimization

At each iteration, we do four things:

1. reset old gradients using `optimizer.zero_grad()`,
2. compute the current loss,
3. call `loss.backward()` so PyTorch computes all parameter gradients,
4. call `optimizer.step()` to update the parameters.

We also store the loss after every iteration so we can plot it later. 

**Note:** You might have to play with `num_iterations` to figure out how many parameter updates you will need.

**VERY IMPORTANT:** Every time you try another `num_iterations`, reinitialize your `a` and `b` by **running the cell above**.

In [ ]:
# TODO 5: run the optimization loop for a suitable number of iterations, 
# and at each iteration compute the loss, perform backpropagation and 
# update the parameters using the optimizer. Store the loss history in 
# a list and print the loss every 10 iterations.

num_iterations = ... 
loss_history = []

for it in range(num_iterations):
    # Your implementation here. 

### Step 8 — Plot the loss versus iteration

This plot shows whether optimization is actually reducing the mismatch between predictions and observations.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel("Iteration", fontsize=13)
plt.ylabel("Loss", fontsize=13)
plt.title("Loss versus iteration", fontsize=15)
plt.grid(True, alpha=0.3)
plt.show()

### Step 9 — Compare the fitted model with the data

Now we compute the fitted trajectory and plot it together with the observed data.


In [ ]:
with torch.no_grad():
    y_pred, t_grid, y_grid = simulate_model(t_obs, y0, a, b, h=h)

plt.figure(figsize=(8, 5))
plt.plot(t_obs, y_obs, "o", label="Observed data")
plt.plot(t_grid, y_grid, label="Fitted model")
plt.xlabel("t", fontsize=13)
plt.ylabel("y", fontsize=13)
plt.title("Observed data and fitted model", fontsize=15)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.show()


### Step 10 — Print the learned coefficients clearly

After optimization, we inspect the recovered values of all ten parameters.


In [ ]:
a_learned = a.detach().clone()
b_learned = b.detach().clone()

for k in range(5):
    print(f"k = {k+1}:  a_{k+1} = {a_learned[k].item(): .6f},   b_{k+1} = {b_learned[k].item(): .6f}")


### Step 11 — Plot the recovered forcing function

The recovered forcing is

$$
g(t)=\sum_{k=1}^{5}\big[a_k\sin(0.7kt)+b_k\cos(0.7kt)\big].
$$

Plotting it helps us see the oscillatory signal that the optimizer learned from the data.


In [ ]:
with torch.no_grad():
    g_grid = forcing_function(t_grid, a_learned, b_learned)

plt.figure(figsize=(8, 5))
plt.plot(t_grid, g_grid)
plt.xlabel("t", fontsize=13)
plt.ylabel("g(t)", fontsize=13)
plt.title("Recovered forcing function", fontsize=15)
plt.grid(True, alpha=0.3)
plt.show()


### Step 12 — Why grid search is completely impractical here

For one parameter, grid search can be reasonable.

For two parameters, grid search is still possible because you can visualize a 2D loss surface.

For ten parameters, grid search becomes hopelessly expensive.

For example, if you tried only 20 candidate values for each parameter, then the total number of combinations would be

$$
20^{10}=10{,}240{,}000{,}000{,}000.
$$

That is more than ten trillion parameter combinations.

For each of those combinations, you would still need to:

1. solve the ODE numerically,
2. compare the prediction with the data,
3. compute the loss.

That is why automatic optimization tools such as PyTorch optimizers become essential in higher-dimensional parameter estimation problems.

## Final checklist

Your completed notebook should include:

- a plot of the observations,
- a correct implementation of the ten-parameter model,
- the least-squares loss,
- the optimization loop,
- a plot of loss versus iteration,
- a plot of observed data and fitted model,
- the learned coefficients,
- a plot of the recovered forcing function.